> **Development note:** This project includes explicit record counts, data previews, and reconciliation checks to make pipeline behavior observable during development and testing. In a production pipeline, unnecessary Spark actions such as repeated `count()` and `display()` calls would be removed or replaced with appropriate monitoring and data-quality metrics to reduce compute overhead.

# LA Parking Citations — Gold Geospatial Enrichment

This notebook implements the **Gold layer** of the LA Parking Citations geospatial data pipeline.

The pipeline is organized into three notebooks and orchestrated as a Databricks Lakeflow Job:

1. **01_ingest_bronze** — Ingest parking citation data from the LA Open Data API and persist the source data to a Bronze Delta table.
2. **02_transform_silver** — Clean, standardize, validate, and quarantine citation records to produce a Silver Delta table.
3. **03_enrich_gold** — Enrich valid citations with City Council district boundaries and create an analytics-ready Gold Delta table.

### Pipeline Architecture

LA Open Data API  
↓  
**Bronze** — Raw parking citation data  
↓  
**Silver** — Cleaned, typed, and validated citations  
↓  
**Gold** — Geospatially enriched citation data

## Notebook 3: Gold Geospatial Enrichment

This notebook reads validated citation records from `parking_citations_silver` and enriches them with Los Angeles City Council district information.

Gold processing includes:

- Loading valid Silver citation records
- Loading City Council district boundary data
- Converting citation coordinates into geospatial points
- Performing a spatial join between citations and council district boundaries
- Validating the enrichment results
- Persisting the analytics-ready Gold Delta table

**Input:** `parking_citations_silver`  
**Output:** `parking_citations_gold`

In [0]:
%pip install geopandas

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import geopandas as gpd

## Load Silver Data

Read the validated Silver Delta table as the source for Gold geospatial enrichment.

In [0]:
silver_df = spark.table("parking_citations_silver")

print(f"Silver records loaded: {silver_df.count():,}")
display(silver_df.limit(10))

Silver records loaded: 49,954


ticket_number,issue_date,issue_time,meter_id,marked_time,rp_state_plate,plate_expiry_date,vin,make,body_style,color,location,route,agency,violation_code,violation_description,fine_amount,agency_desc,color_desc,body_style_desc,loc_lat,loc_long,geocodelocation,issue_time_formatted,valid_coordinates,valid_ticket_number,valid_issue_date,valid_issue_time,valid_fine_amount,valid_violation_code,quarantine_reason
4603856084,2025-05-31T00:00:00.000Z,2113,null,0000,CA,202408,null,JEEP,PA,GY,850 7TH ST W,null,56,5200,DISPLAY OF PLATES,25.0,56 - DOT - CENTRAL,GREY,PASSENGER CAR,34.0390366,-118.2447298,"{'type': 'Point', 'coordinates': [-118.2447298, 34.0390366]}",21:13,true,true,true,true,true,true,null
4603930730,2025-05-31T00:00:00.000Z,2134,null,0000,CA,202501,null,MERZ,PA,GY,5315 9TH ST W,null,51,80.58L,PREFERENTIAL PARKING,68.0,51 - DOT - WESTERN,GREY,PASSENGER CAR,34.05878999,-118.34604232,"{'type': 'Point', 'coordinates': [-118.34604232, 34.05878999]}",21:34,true,true,true,true,true,true,null
4604402014,2025-05-31T00:00:00.000Z,1456,null,0000,CA,202302,null,PORS,PA,GY,146 LAUREL AV S,null,54,4000(a)1,EXPIRED REGISTRATION,50.0,54 - DOT - HOLLYWOOD,GREY,PASSENGER CAR,34.07277029,-118.36464923,"{'type': 'Point', 'coordinates': [-118.36464923, 34.07277029]}",14:56,true,true,true,true,true,true,null
4604155253,2025-05-31T00:00:00.000Z,1659,SO14A,0000,VT,202407,null,JEEP,PA,BK,14501 VENTURA BL,null,53,88.13B+,METER EXP.,63.0,53 - DOT - VALLEY,BLACK,PASSENGER CAR,34.1513168,-118.450474,"{'type': 'Point', 'coordinates': [-118.450474, 34.1513168]}",16:59,true,true,true,true,true,true,null
4603930601,2025-05-31T00:00:00.000Z,1823,null,0000,CA,202504,null,TOYT,PA,BK,744 DUNSMUIR AV,null,51,80.58L,PREFERENTIAL PARKING,68.0,51 - DOT - WESTERN,BLACK,PASSENGER CAR,34.06100693,-118.34874459,"{'type': 'Point', 'coordinates': [-118.34874459, 34.06100693]}",18:23,true,true,true,true,true,true,null
4604347801,2025-05-31T00:00:00.000Z,1447,null,0000,CA,202601,null,MAZD,PA,BK,133 FLORES ST S,null,54,5200,DISPLAY OF PLATES,25.0,54 - DOT - HOLLYWOOD,BLACK,PASSENGER CAR,34.07349098,-118.37118875,"{'type': 'Point', 'coordinates': [-118.37118875, 34.07349098]}",14:47,true,true,true,true,true,true,null
4603783671,2025-05-31T00:00:00.000Z,1455,null,0000,CA,202601,null,AUDI,PA,BK,665 MARIPOSA AV S,null,54,80.56E4+,RED ZONE,93.0,54 - DOT - HOLLYWOOD,BLACK,PASSENGER CAR,34.0614794,-118.2988131,"{'type': 'Point', 'coordinates': [-118.2988131, 34.0614794]}",14:55,true,true,true,true,true,true,null
4604015021,2025-05-31T00:00:00.000Z,1915,null,0000,CA,202509,null,TOYT,PA,WT,4173 DIXIE CANYON AV,null,53,80.58L,PREFERENTIAL PARKING,68.0,53 - DOT - VALLEY,WHITE,PASSENGER CAR,34.1459983,-118.4249253,"{'type': 'Point', 'coordinates': [-118.4249253, 34.1459983]}",19:15,true,true,true,true,true,true,null
4604433750,2025-05-31T00:00:00.000Z,1624,SS128,0000,CA,202510,null,BMW,PA,GY,11300 IDAHO AV,null,51,88.13B+,METER EXP.,63.0,51 - DOT - WESTERN,GREY,PASSENGER CAR,34.04570223,-118.44838483,"{'type': 'Point', 'coordinates': [-118.44838483, 34.04570223]}",16:24,true,true,true,true,true,true,null
4604016244,2025-05-31T00:00:00.000Z,1858,null,0000,CA,202501,null,NISS,PA,OT,11300 GLENOAKS BLVD,null,53,80.61,STANDNG IN ALLEY,68.0,53 - DOT - VALLEY,OTHER,PASSENGER CAR,34.2751736,-118.41408033,"{'type': 'Point', 'coordinates': [-118.41408033, 34.2751736]}",18:58,true,true,true,true,true,true,null


In [0]:
silver_df.printSchema()

root
 |-- ticket_number: string (nullable = true)
 |-- issue_date: timestamp (nullable = true)
 |-- issue_time: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- marked_time: string (nullable = true)
 |-- rp_state_plate: string (nullable = true)
 |-- plate_expiry_date: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- make: string (nullable = true)
 |-- body_style: string (nullable = true)
 |-- color: string (nullable = true)
 |-- location: string (nullable = true)
 |-- route: string (nullable = true)
 |-- agency: integer (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- fine_amount: double (nullable = true)
 |-- agency_desc: string (nullable = true)
 |-- color_desc: string (nullable = true)
 |-- body_style_desc: string (nullable = true)
 |-- loc_lat: double (nullable = true)
 |-- loc_long: double (nullable = true)
 |-- geocodelocation: string (nullable = true)
 |-- issue_time_fo

## Prepare Geospatial Dependencies

GeoPandas is used in the Gold layer to create citation point geometry and perform point-in-polygon matching against Los Angeles City Council district boundaries.

## Load City Council District Boundaries

Retrieve Los Angeles City Council district boundaries from the LA City ArcGIS API. These polygon boundaries will be used to assign each parking citation to its corresponding council district.

In [0]:
import requests

city_council_url = (
    "https://maps.lacity.org/lahub/rest/services/"
    "Boundaries/MapServer/13/query"
    "?where=1%3D1"
    "&outFields=*"
    "&outSR=4326"
    "&f=json"
)

district_response = requests.get(city_council_url)
district_response.raise_for_status()

raw_districts = district_response.json()

In [0]:
# Check
print(f"Council districts loaded: {len(raw_districts['features'])}")
print(raw_districts["features"][0].keys())

Council districts loaded: 15
dict_keys(['attributes', 'geometry'])


## Create the District GeoDataFrame

Convert the ArcGIS response into a GeoPandas GeoDataFrame so the district polygon geometries can be used for spatial operations.

In [0]:
import json
from io import StringIO

districts_gdf = gpd.read_file(
    StringIO(json.dumps(raw_districts))
)

In [0]:
# Check
districts_gdf.head()

,OBJECTID,NAME,District,District_Name,TOOLTIP,NLA_URL,geometry
0,1,Eunisses Hernandez,1,1 - Eunisses Hernandez,Councilmember: Eunisses Hernandez\nDistrict: 1,https://councildistrict1.lacity.gov/?nla_win=p...,"POLYGON ((-118.2687 34.04671, -118.2689 34.046..."
1,2,Adrin Nazarian,2,2 - Adrin Nazarian,Councilmember: Adrin Nazarian\nDistrict: 2,https://councildistrict2.lacity.gov/?nla_win=p...,"POLYGON ((-118.33733 34.22104, -118.33456 34.2..."
2,3,Bob Blumenfield,3,3 - Bob Blumenfield,Councilmember: Bob Blumenfield\nDistrict: 3,https://councildistrict3.lacity.gov/?nla_win=p...,"POLYGON ((-118.5492 34.22053, -118.5492 34.220..."
3,4,Nithya Raman,4,4 - Nithya Raman,Councilmember: Nithya Raman\nDistrict: 4,https://councildistrict4.lacity.gov/?nla_win=p...,"POLYGON ((-118.51861 34.20841, -118.5186 34.20..."
4,5,Katy Yaroslavsky,5,5 - Katy Yaroslavsky,Councilmember: Katy Yaroslavsky\nDistrict: 5,https://councildistrict5.lacity.gov/?nla_win=p...,"POLYGON ((-118.3916 34.11243, -118.39231 34.11..."


In [0]:
print("CRS:", districts_gdf.crs)
print("Geometry types:")
print(districts_gdf.geometry.geom_type.value_counts())

CRS: EPSG:4326
Geometry types:
Polygon    15
Name: count, dtype: int64


## Create Citation Point Geometry

Convert the validated Silver citation coordinates into geospatial point geometry.

Each citation point is created from its longitude and latitude using EPSG:4326, matching the coordinate reference system of the City Council district boundaries.

In [0]:
# convert to DF, create geodataframe
parking_pdf = silver_df.toPandas()

In [0]:
parking_gdf = gpd.GeoDataFrame(
    parking_pdf,
    geometry=gpd.points_from_xy(
        parking_pdf["loc_long"],
        parking_pdf["loc_lat"]
    ),
    crs="EPSG:4326"
)

## Enrich Citations with City Council Districts

Perform a point-in-polygon spatial join to assign each parking citation to the Los Angeles City Council district containing its geographic location.

A left join preserves all valid Silver citation records, including any citations that do not match a council district boundary.

In [0]:
gold_gdf = gpd.sjoin(
    parking_gdf,
    districts_gdf[
        [
            "District",
            "District_Name",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

In [0]:
# CHECK
print(f"Gold records after spatial join: {len(gold_gdf):,}")
print(f"Unmatched council districts: {gold_gdf['District'].isna().sum():,}")

gold_gdf[
    [
        "ticket_number",
        "location",
        "District",
        "District_Name"
    ]
].head(10)

Gold records after spatial join: 49,954
Unmatched council districts: 179


,ticket_number,location,District,District_Name
0,4603856084,850 7TH ST W,14.0,14 - Ysabel J. Jurado
1,4603930730,5315 9TH ST W,5.0,5 - Katy Yaroslavsky
2,4604402014,146 LAUREL AV S,5.0,5 - Katy Yaroslavsky
3,4604155253,14501 VENTURA BL,4.0,4 - Nithya Raman
4,4603930601,744 DUNSMUIR AV,5.0,5 - Katy Yaroslavsky
5,4604347801,133 FLORES ST S,5.0,5 - Katy Yaroslavsky
6,4603783671,665 MARIPOSA AV S,10.0,10 - Heather Hutt
7,4604015021,4173 DIXIE CANYON AV,4.0,4 - Nithya Raman
8,4604433750,11300 IDAHO AV,11.0,11 - Traci Park
9,4604016244,11300 GLENOAKS BLVD,7.0,7 - Monica Rodriguez


## Prepare Gold Data for Persistence

Remove GeoPandas-specific columns and prepare the enriched citation data for conversion back to a Spark DataFrame.

In [0]:
gold_pdf = gold_gdf.drop(
    columns=["geometry", "index_right"],
    errors="ignore"
).copy()

gold_pdf["District"] = gold_pdf["District"].astype("Int64")
gold_pdf["agency"] = gold_pdf["agency"].astype("Int64")

gold_pdf["loc_lat"] = gold_pdf["loc_lat"].astype("float64")
gold_pdf["loc_long"] = gold_pdf["loc_long"].astype("float64")

## Convert Gold Data Back to Spark

Convert the enriched pandas DataFrame back into a Spark DataFrame so the Gold layer can be validated and persisted as a Delta table.

In [0]:
gold_df = spark.createDataFrame(gold_pdf)

In [0]:
gold_df.printSchema()

root
 |-- ticket_number: string (nullable = true)
 |-- issue_date: timestamp (nullable = true)
 |-- issue_time: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- marked_time: string (nullable = true)
 |-- rp_state_plate: string (nullable = true)
 |-- plate_expiry_date: string (nullable = true)
 |-- vin: void (nullable = true)
 |-- make: string (nullable = true)
 |-- body_style: string (nullable = true)
 |-- color: string (nullable = true)
 |-- location: string (nullable = true)
 |-- route: string (nullable = true)
 |-- agency: long (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- fine_amount: double (nullable = true)
 |-- agency_desc: string (nullable = true)
 |-- color_desc: string (nullable = true)
 |-- body_style_desc: string (nullable = true)
 |-- loc_lat: double (nullable = true)
 |-- loc_long: double (nullable = true)
 |-- geocodelocation: string (nullable = true)
 |-- issue_time_formatt

In [0]:
from pyspark.sql import functions as F

In [0]:
gold_df = gold_df.withColumn(
    "vin",
    F.col("vin").cast("string")
)

## Persist Gold Layer

Persist the geospatially enriched citation records as the analytics-ready Gold Delta table.

In [0]:
spark.sql("DROP TABLE IF EXISTS parking_citations_gold")

DataFrame[]

In [0]:
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("parking_citations_gold")
)

In [0]:
gold_table = spark.table("parking_citations_gold")

print(f"Gold records: {gold_table.count():,}")

display(
    gold_table.select(
        "ticket_number",
        "issue_date",
        "location",
        "violation_description",
        "fine_amount",
        "District",
        "District_Name"
    ).limit(10)
)

Gold records: 49,954


ticket_number,issue_date,location,violation_description,fine_amount,District,District_Name
4604674491,2025-06-05T00:00:00.000Z,6658 WOODMAN AVE,EXPIRED REGISTRATION,50.0,2,2 - Adrin Nazarian
4604684641,2025-06-05T00:00:00.000Z,625 LANDFAIR AVE,NO PARK/STREET CLEAN,73.0,5,5 - Katy Yaroslavsky
4603959006,2025-06-05T00:00:00.000Z,20700 BRYANT ST,NO PARK/STREET CLEAN,73.0,3,3 - Bob Blumenfield
4604675740,2025-06-05T00:00:00.000Z,3118 LA SALLE AVE S,NO PARK/STREET CLEAN,73.0,8,8 - Marqueece Harris-Dawson
4604443071,2025-06-05T00:00:00.000Z,3795 ROSEWOOD AV,NO PARK/STREET CLEAN,73.0,11,11 - Traci Park
4604637715,2025-06-05T00:00:00.000Z,1330 35TH PL W,NO PARK/STREET CLEAN,73.0,8,8 - Marqueece Harris-Dawson
4604657960,2025-06-05T00:00:00.000Z,5216 ROSELAND ST,PREFERENTIAL PARKING,68.0,10,10 - Heather Hutt
4604096151,2025-06-05T00:00:00.000Z,1701 CHEROKEE AV,METER EXP.,63.0,13,13 - Hugo Soto-Martinez
4604217914,2025-06-03T00:00:00.000Z,3120 71ST ST W,NO PARK/STREET CLEAN,73.0,8,8 - Marqueece Harris-Dawson
4604735450,2025-06-05T00:00:00.000Z,1144 39TH ST W,NO PARK/STREET CLEAN,73.0,8,8 - Marqueece Harris-Dawson
